In [0]:
from pyspark.sql import functions as F
from datetime import datetime, UTC
import uuid

gold_train = spark.table("airlinepassengers.airlinedata.silver_train")
gold_test = spark.table("airlinepassengers.airlinedata.silver_test")

In [0]:
business_kpi_df = gold_train.agg(
    F.count("*").alias("total_passengers"),
    F.sum(
        F.when(F.col("satisfaction") == "satisfied", 1).otherwise(0)
    ).alias("satisfied_passengers"),
    F.sum(
        F.when(F.col("satisfaction") == "neutral or dissatisfied", 1).otherwise(0)
    ).alias("unsatisfied_passengers"),
    F.avg("flight_distance").alias("average_flight_distance"),
    F.avg("departure_delay_in_minutes").alias("average_departure_delay"),
    F.avg("arrival_delay_in_minutes").alias("average_arrival_delay"),
    F.avg("age").alias("average_age")
)

display(business_kpi_df)

total_passengers,satisfied_passengers,unsatisfied_passengers,average_flight_distance,average_departure_delay,average_arrival_delay,average_age
103904,45025,58879,1189.4483754234677,14.815618263012011,15.178678301832152,39.379706267323684


In [0]:
summary_table_df = gold_train.groupBy(
    "class",
    "satisfaction"
).agg(
    F.count("*").alias("total_passengers")
).orderBy("class")

display(summary_table_df)

class,satisfaction,total_passengers
Business,neutral or dissatisfied,15185
Business,satisfied,34480
Eco,neutral or dissatisfied,38044
Eco,satisfied,8701
Eco Plus,neutral or dissatisfied,5650
Eco Plus,satisfied,1844


In [0]:
trend_analysis_df = gold_train.groupBy(
    "type_of_travel"
).agg(
    F.avg("departure_delay_in_minutes").alias("avg_departure_delay"),
    F.avg("arrival_delay_in_minutes").alias("avg_arrival_delay"),
    F.count("*").alias("passenger_count")
)
display(trend_analysis_df)

type_of_travel,avg_departure_delay,avg_arrival_delay,passenger_count
Personal Travel,14.506062203479178,14.850664508699307,32249
Business travel,14.954936850184914,15.326145665710488,71655


In [0]:
performance_metrics_df = gold_train.groupBy(
    "gender"
).agg(
    F.avg("inflight_wifi_service").alias("avg_wifi"),
    F.avg("seat_comfort").alias("avg_seat_comfort"),
    F.avg("food_and_drink").alias("avg_food"),
    F.avg("cleanliness").alias("avg_cleanliness"),
    F.avg("online_boarding").alias("avg_online_boarding")
)
display(performance_metrics_df)

gender,avg_wifi,avg_seat_comfort,avg_food,avg_cleanliness,avg_online_boarding
Female,2.7179433686725964,3.473836933639312,3.194568247766799,3.277941092798756,3.3065602063458948
Male,2.741778533325517,3.4039119135549174,3.2099185180842955,3.295015338921781,3.1924888133341147


In [0]:
reporting_table_df = gold_train.select(
    "id","gender","customer_type","class","type_of_travel","flight_distance","departure_delay_in_minutes",
    "arrival_delay_in_minutes", "cleanliness","seat_comfort", "satisfaction"
)
display(reporting_table_df)

id,gender,customer_type,class,type_of_travel,flight_distance,departure_delay_in_minutes,arrival_delay_in_minutes,cleanliness,seat_comfort,satisfaction
70172,Male,Loyal Customer,Eco Plus,Personal Travel,460,25,18.0,5,5,neutral or dissatisfied
5047,Male,disloyal Customer,Business,Business travel,235,1,6.0,1,1,neutral or dissatisfied
110028,Female,Loyal Customer,Business,Business travel,1142,0,0.0,5,5,satisfied
24026,Female,Loyal Customer,Business,Business travel,562,11,9.0,2,2,neutral or dissatisfied
119299,Male,Loyal Customer,Business,Business travel,214,0,0.0,3,5,satisfied
111157,Female,Loyal Customer,Eco,Personal Travel,1180,0,0.0,1,1,neutral or dissatisfied
82113,Male,Loyal Customer,Eco,Personal Travel,1276,9,23.0,2,2,neutral or dissatisfied
96462,Female,Loyal Customer,Business,Business travel,2035,4,0.0,4,5,satisfied
79485,Female,Loyal Customer,Business,Business travel,853,0,0.0,2,3,neutral or dissatisfied
65725,Male,disloyal Customer,Eco,Business travel,1061,0,0.0,2,3,neutral or dissatisfied


In [0]:
business_kpi_df.write \
.format("delta") \
.mode("overwrite") \
.saveAsTable("airlinepassengers.airlinedata.gold_business_kpi")

summary_table_df.write \
.format("delta") \
.mode("overwrite") \
.saveAsTable("airlinepassengers.airlinedata.gold_summary")

trend_analysis_df.write \
.format("delta") \
.mode("overwrite") \
.saveAsTable("airlinepassengers.airlinedata.gold_trend_analysis")

performance_metrics_df.write \
.format("delta") \
.mode("overwrite") \
.saveAsTable("airlinepassengers.airlinedata.gold_performance_metrics")

reporting_table_df.write \
.format("delta") \
.mode("overwrite") \
.saveAsTable("airlinepassengers.airlinedata.gold_reporting")

In [0]:
display(spark.table("airlinepassengers.airlinedata.gold_business_kpi"))
display(spark.table("airlinepassengers.airlinedata.gold_summary"))
display(spark.table("airlinepassengers.airlinedata.gold_trend_analysis"))
display(spark.table("airlinepassengers.airlinedata.gold_performance_metrics"))
display(spark.table("airlinepassengers.airlinedata.gold_reporting"))

total_passengers,satisfied_passengers,unsatisfied_passengers,average_flight_distance,average_departure_delay,average_arrival_delay,average_age
103904,45025,58879,1189.4483754234677,14.815618263012011,15.178678301832152,39.379706267323684


class,satisfaction,total_passengers
Business,neutral or dissatisfied,15185
Business,satisfied,34480
Eco,neutral or dissatisfied,38044
Eco,satisfied,8701
Eco Plus,neutral or dissatisfied,5650
Eco Plus,satisfied,1844


type_of_travel,avg_departure_delay,avg_arrival_delay,passenger_count
Personal Travel,14.506062203479178,14.850664508699307,32249
Business travel,14.954936850184914,15.326145665710488,71655


gender,avg_wifi,avg_seat_comfort,avg_food,avg_cleanliness,avg_online_boarding
Female,2.7179433686725964,3.473836933639312,3.194568247766799,3.277941092798756,3.3065602063458948
Male,2.741778533325517,3.4039119135549174,3.2099185180842955,3.295015338921781,3.1924888133341147


id,gender,customer_type,class,type_of_travel,flight_distance,departure_delay_in_minutes,arrival_delay_in_minutes,cleanliness,seat_comfort,satisfaction
70172,Male,Loyal Customer,Eco Plus,Personal Travel,460,25,18.0,5,5,neutral or dissatisfied
5047,Male,disloyal Customer,Business,Business travel,235,1,6.0,1,1,neutral or dissatisfied
110028,Female,Loyal Customer,Business,Business travel,1142,0,0.0,5,5,satisfied
24026,Female,Loyal Customer,Business,Business travel,562,11,9.0,2,2,neutral or dissatisfied
119299,Male,Loyal Customer,Business,Business travel,214,0,0.0,3,5,satisfied
111157,Female,Loyal Customer,Eco,Personal Travel,1180,0,0.0,1,1,neutral or dissatisfied
82113,Male,Loyal Customer,Eco,Personal Travel,1276,9,23.0,2,2,neutral or dissatisfied
96462,Female,Loyal Customer,Business,Business travel,2035,4,0.0,4,5,satisfied
79485,Female,Loyal Customer,Business,Business travel,853,0,0.0,2,3,neutral or dissatisfied
65725,Male,disloyal Customer,Eco,Business travel,1061,0,0.0,2,3,neutral or dissatisfied


In [0]:
from pyspark.sql import functions as F
from pyspark.sql import Row

validation_results = []

# 1. Aggregate Totals
silver_total = gold_train.count()
gold_total = reporting_table_df.count()
validation_results.append(
    Row(
        validation_name="Aggregate Totals",
        expected=silver_total,
        actual=gold_total,
        status="PASS" if silver_total == gold_total else "FAIL"
    )
)

# 2. Data Completeness
null_count = reporting_table_df.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c)
    for c in reporting_table_df.columns
]).collect()[0]

total_nulls = sum(null_count)

validation_results.append(
    Row(
        validation_name="Data Completeness",
        expected=0,
        actual=total_nulls,
        status="PASS" if total_nulls == 0 else "FAIL"
    )
)

# 3. Silver vs Gold Consistency
silver_satisfied = gold_train.filter(
    F.col("satisfaction") == "satisfied"
).count()

gold_satisfied = reporting_table_df.filter(
    F.col("satisfaction") == "satisfied"
).count()

validation_results.append(
    Row(
        validation_name="Silver-Gold Consistency",
        expected=silver_satisfied,
        actual=gold_satisfied,
        status="PASS" if silver_satisfied == gold_satisfied else "FAIL"
    )
)

# 4. Duplicate Aggregates
duplicates = reporting_table_df.groupBy("id") \
    .count() \
    .filter(F.col("count") > 1) \
    .count()

validation_results.append(
    Row(
        validation_name="Duplicate Aggregates",
        expected=0,
        actual=duplicates,
        status="PASS" if duplicates == 0 else "FAIL"
    )
)

# 5. Missing Dimensions
dimension_columns = [
    "gender",
    "customer_type",
    "class",
    "type_of_travel"
]

missing_dimensions = 0

for c in dimension_columns:
    missing_dimensions += reporting_table_df.filter(
        F.col(c).isNull()
    ).count()

validation_results.append(
    Row(
        validation_name="Missing Dimensions",
        expected=0,
        actual=missing_dimensions,
        status="PASS" if missing_dimensions == 0 else "FAIL"
    )
)

# 6. Missing Measures
measure_columns = [
    "flight_distance",
    "departure_delay_in_minutes",
    "arrival_delay_in_minutes",
    "seat_comfort",
    "cleanliness"
]

missing_measures = 0

for c in measure_columns:
    missing_measures += reporting_table_df.filter(
        F.col(c).isNull()
    ).count()

validation_results.append(
    Row(
        validation_name="Missing Measures",
        expected=0,
        actual=missing_measures,
        status="PASS" if missing_measures == 0 else "FAIL"
    )
)

# Validation Report
gold_validation_df = spark.createDataFrame(validation_results)

display(gold_validation_df)

validation_name,expected,actual,status
Aggregate Totals,103904,103904,PASS
Data Completeness,0,310,FAIL
Silver-Gold Consistency,45025,45025,PASS
Duplicate Aggregates,0,0,PASS
Missing Dimensions,0,0,PASS
Missing Measures,0,310,FAIL


In [0]:
gold_validation_df.write \
    .format("delta") \
    .mode("append") \
    .saveAsTable("airlinepassengers.airlinedata.gold_validation_report")

In [0]:
from pyspark.sql import Row
from datetime import datetime, UTC
import uuid
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, FloatType, TimestampType

run_id = str(uuid.uuid4())
job_start_time = datetime.now(UTC)

try:
    # Source Records
    source_records = gold_train.count()

    # Target Records
    target_records = reporting_table_df.count()

    # Aggregation Counts
    aggregation_count = (
        business_kpi_df.count()
        + summary_table_df.count()
        + trend_analysis_df.count()
        + performance_metrics_df.count()
    )

    # KPI Generation Status
    kpi_generation_status = "SUCCESS" if business_kpi_df.count() > 0 else "FAILED"

    processing_status = "SUCCESS"
    error_message = None

except Exception as e:
    source_records = 0
    target_records = 0
    aggregation_count = 0
    kpi_generation_status = "FAILED"
    processing_status = "FAILED"
    error_message = str(e)

job_end_time = datetime.now(UTC)

processing_duration = (job_end_time - job_start_time).total_seconds()

schema = StructType([
    StructField('run_id', StringType(), True),
    StructField('job_start_time', TimestampType(), True),
    StructField('job_end_time', TimestampType(), True),
    StructField('processing_duration_seconds', FloatType(), True),
    StructField('source_records', IntegerType(), True),
    StructField('target_records', IntegerType(), True),
    StructField('aggregation_count', IntegerType(), True),
    StructField('kpi_generation_status', StringType(), True),
    StructField('processing_status', StringType(), True),
    StructField('error_message', StringType(), True)
])

gold_log_df = spark.createDataFrame([
    Row(
        run_id=run_id,
        job_start_time=job_start_time,
        job_end_time=job_end_time,
        processing_duration_seconds=processing_duration,
        source_records=source_records,
        target_records=target_records,
        aggregation_count=aggregation_count,
        kpi_generation_status=kpi_generation_status,
        processing_status=processing_status,
        error_message=error_message
    )
], schema=schema)

display(gold_log_df)

run_id,job_start_time,job_end_time,processing_duration_seconds,source_records,target_records,aggregation_count,kpi_generation_status,processing_status,error_message
6ad8f5c7-c4c0-486f-a876-1f163d38fc1b,2026-06-26T17:05:00.985237Z,2026-06-26T17:05:03.554204Z,2.568967,103904,103904,11,SUCCESS,SUCCESS,null


In [0]:

### Audit Requirements
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, FloatType, TimestampType, DateType

schema = StructType([
    StructField('batch_id', StringType(), True),
    StructField('aggregation_date', DateType(), True),
    StructField('processing_timestamp', TimestampType(), True),
    StructField('gold_load_status_code', StringType(), True),

    StructField('run_id', StringType(), True),
    StructField('job_start_time', TimestampType(), True),
    StructField('job_end_time', TimestampType(), True),
    StructField('processing_duration_seconds', FloatType(), True),
    StructField('source_records', IntegerType(), True),
    StructField('target_records', IntegerType(), True),
    StructField('aggregation_count', IntegerType(), True),
    StructField('kpi_generation_status', StringType(), True),
    StructField('processing_status', StringType(), True),
    StructField('error_message', StringType(), True)
])

In [0]:
from datetime import datetime

batch_id = str(uuid.uuid4())
aggregation_date = job_start_time.date()
processing_timestamp = datetime.now(UTC)
gold_load_status_code = "SUCCESS" if processing_status == "SUCCESS" else "FAILED"
gold_log_df = spark.createDataFrame([
    Row(
        batch_id=batch_id,
        aggregation_date=aggregation_date,
        processing_timestamp=processing_timestamp,
        gold_load_status_code=gold_load_status_code,

        run_id=run_id,
        job_start_time=job_start_time,
        job_end_time=job_end_time,
        processing_duration_seconds=processing_duration,
        source_records=source_records,
        target_records=target_records,
        aggregation_count=aggregation_count,
        kpi_generation_status=kpi_generation_status,
        processing_status=processing_status,
        error_message=error_message
    )
], schema=schema)